<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
Installing Dependencies
</h2>
</div>

In [1]:
!pip install -U peft bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 100.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 36.3 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 30.3 MB/s eta 0:00:00


<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
    Loading Dataset
</h2>
</div>

In [4]:
from datasets import Dataset, load_dataset

In [5]:
from datasets import load_dataset

# Load only the train split
dataset = load_dataset(
    "AnmolNimmala0/agri-slm-corpus",
    split="train"
)

# Randomly select 20k examples
dataset = dataset.shuffle(seed=42).select(range(10000))

print(dataset)

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


corpus_filtered.jsonl:   0%|          | 0.00/6.84G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text', 'source', 'domain', 'subdomain', 'language', 'word_count', 'url', 'title', 'year', 'journal', 'crop', 'state'],
    num_rows: 10000
})


In [6]:
print(dataset)

Dataset({
    features: ['text', 'source', 'domain', 'subdomain', 'language', 'word_count', 'url', 'title', 'year', 'journal', 'crop', 'state'],
    num_rows: 10000
})


In [7]:
print(dataset[5]["text"])

Plant Biostimulants from Cyanobacteria: An Emerging Strategy to Improve Yields and Sustainability in Agriculture

Cyanobacteria can be considered a promising source for the development of new biostimulants as they are known to produce a variety of biologically active molecules that can positively affect plant growth, nutrient use efficiency, qualitative traits of the final product, and increase plant tolerance to abiotic stresses. Moreover, the cultivation of cyanobacteria in controlled and confined systems, along with their metabolic plasticity, provides the possibility to improve and standardize composition and effects on plants of derived biostimulant extracts or hydrolysates, which is one of the most critical aspects in the production of commercial biostimulants. Faced with these opportunities, research on biostimulant properties of cyanobacteria has undergone a significant growth in recent years. However, research in this field is still scarce, especially as regards the number of 

#### Removing Extra Columns

In [8]:
dataset = dataset.remove_columns(
    [col for col in dataset.column_names if col != "text"]
)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 10000
})


In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [23]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
Loading Tokenizer from base model
</h2>
</div>

In [24]:
tokenizer = AutoTokenizer.from_pretrained(model)

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
Checks Wheather base model have padding token or not
</h2>
</div>

In [25]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
Tokenization Function for Creating Training Inputs and Labels in SFT
</h2>
</div>

In [26]:
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [27]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
Loading TinyLlama Model with 8-bit Quantization for Memory-Efficient Fine-Tuning
</h2>
</div>

In [28]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
Setting Up Parameter-Efficient Fine-Tuning (LoRA) Configuration
</h2>
</div>

In [29]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [30]:
# save adapter
non_inst_model_lora = get_peft_model(model, lora_config)

<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
    Training Arguments Configuration for TinyLlama Agriculture LoRA Non Instruction Fine-Tuning
</h2>
</div>

In [31]:
args = TrainingArguments(
    output_dir="./tinyllama-aggriculture-domain-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
    Initializing Trainer for Non Instruction Fine-Tuning
</h2>
</div>

In [32]:
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=tokenized
)

<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
Training the Non Instruction Fine-Tuning Model
</h2>
</div>

In [33]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
20,2.853680
40,2.195764
60,1.864026
80,1.777067
100,1.792593
120,1.742050
140,1.778018
160,1.726847
180,1.722037
200,1.758878


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during q

TrainOutput(global_step=6250, training_loss=1.723194436340332, metrics={'train_runtime': 29794.036, 'train_samples_per_second': 1.678, 'train_steps_per_second': 0.21, 'total_flos': 1.590741172224e+17, 'train_loss': 1.723194436340332, 'epoch': 5.0})

In [41]:
model_path ="./tinyllama-aggriculture-domain-lora/checkpoint-6250"

In [43]:
pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 66.4 MB/s eta 0:00:00:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
    Loading Saved Trained Non Instruction Fine-Tuned Model
</h2>
</div>

In [44]:
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [54]:
print(dataset[3]["text"])

Development of a PCR-based, genetic marker resource for the tomato-like nightshade relative, Solanum lycopersicoides using whole genome sequence analysis.

Solanum lycopersicoides is a wild nightshade relative of tomato with known resistance to a wide range of pests and pathogens, as well as tolerance to cold, drought and salt stress. To effectively utilize S. lycopersicoides as a genetic resource in breeding for tomato improvement, the underlying basis of observable traits in the species needs to be understood. Molecular markers are important tools that can unlock the genetic underpinnings of phenotypic variation in wild crop relatives. Unfortunately, DNA markers that are specific to S. lycopersicoides are limited in number, distribution and polymorphism rate. In this study, we developed a suite of S. lycopersicoides-specific SSR and indel markers by sequencing, building and analyzing a draft assembly of the wild nightshade genome. Mapping of a total of 1.45 Gb of S. lycopersicoides c

<div style="background-color:#E8F8F5; padding:20px; border-radius:10px; text-align:center;">
<h2 style="color:#117864;">
    Response Generation by Trained Non Instruction Fine-Tuned Model
</h2>
</div>

In [55]:
prompt ="""Development of a PCR-based, genetic marker resource for the tomato-like nightshade relative, Solanum lycopersicoides using 
whole genome sequence analysis"""

In [56]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [57]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [58]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Development of a PCR-based, genetic marker resource for the tomato-like nightshade relative, Solanum lycopersicoides using 
whole genome sequence analysis.

<h4>Background</h4>Solanum lycopersicum (tomato) is an important crop species with numerous biological and agronomic applications, and its wild relatives are also valuable resources for sustainable agricultural production. However, there are currently no cultivated relatives for S. lycopersicum that have been extensively characterized for their traits. Therefore, we developed a novel genetic marker resource based on the 16S


#### Creating Zip of Non Instruction Fined-Tuned Model and Saving it

In [62]:
import shutil

shutil.make_archive(
    "tinyllama-aggriculture-domain-lora",
    "zip",
    "/kaggle/working/tinyllama-aggriculture-domain-lora"
)

'/kaggle/working/tinyllama-aggriculture-domain-lora.zip'